# **Hyperparameter Optimization with Traditional ML Models**
This notebook performs hyperparameter optimization for a range of traditional machine learning regressors using GridSearchCV on the training set. The following models are tuned and evaluated:

- Random Forest Regressor

- XGBoost Regressor

- Gradient Boosting Regressor

- Support Vector Regressor (SVR)

- K-Nearest Neighbors Regressor

- Multi-Layer Perceptron Regressor (MLP)

**Workflow:**
1. Training and Test Splits: Uses predefined train/test splits for a given target (e.g., 5-HT6) with REU-selected features.

2. Scaling: Applies feature scaling using StandardScaler for models sensitive to feature magnitudes (SVR, KNN, MLP).

3. Grid Search (Inner Loop): Hyperparameters are optimized using 5-fold cross-validation on the training set.

4. Model Evaluation:

  - Predictions are made on the training set (used in CV).

  - Predictions are also made on the held-out test set to assess generalization.

**Results:**
  - Performance metrics (R² and RMSE) are recorded for both training and test sets.

  - The best parameters and test metrics are saved for each model.


**Key Notes**:
- Models like SVR, KNN, and MLP include a StandardScaler step in the pipeline.

- Grid search is performed only on training data (5-fold CV).

- The test set is not used during hyperparameter tuning—only for final evaluation.

- Overfitting may occur since there's no nested CV.



In [17]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from math import sqrt
import warnings
import os

In [19]:
warnings.filterwarnings("ignore")

In [20]:
# === Paths to datasets ===

reu_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/descriptors/reduced_features_reu_2"
meta_cols = ['molecule_chembl_id', 'canonical_smiles', 'activity_class', 'pIC50']

In [21]:
targets = ["5-HT6","ache", "bace1", "buche","esr1","gsk-3beta", "mao-b"]

target=targets[0]

In [22]:
#df_train = pd.read_csv(f"{reu_path}/{target}_train_reu.csv").dropna(how="any")
#df_test = pd.read_csv(f"{reu_path}/{target}_test_reu.csv").dropna(how="any")

df_train = pd.read_csv(f"{reu_path}/{target}_train_reu_topN.csv").dropna(how="any")
df_test = pd.read_csv(f"{reu_path}/{target}_test_reu_topN.csv").dropna(how="any")

In [23]:
df_train.head()

,molecule_chembl_id,canonical_smiles,activity_class,pIC50,fp_935,fp_883,fp_881,fp_879,fp_875,fp_867,fp_847,fp_843,fp_841,fp_885,fp_934,fp_932,fp_926
0,CHEMBL267615,CCc1[nH]c2ccc(OC)cc2c1CCN(C)C,active,7.070581,0,0,1,0,1,0,0,0,1,0,0,0,0
1,CHEMBL5169546,COc1ccc(S(=O)(=O)n2ccc3ccc(Cl)cc32)cc1N1CCNCC1,active,8.443697,1,0,0,0,1,0,0,0,1,0,0,0,1
2,CHEMBL3398397,CN(C)C=NN=Cc1cn(S(=O)(=O)c2cccc3ccccc23)c2cccc...,inactive,5.528708,1,1,1,0,0,0,0,0,0,0,0,0,0
3,CHEMBL4742804,CC(C)c1nc2c(N3CCNCC3)nccc2n1S(=O)(=O)c1cccc(Cl)c1,active,8.283997,1,0,0,0,1,0,0,0,0,0,0,1,1
4,CHEMBL1922635,CNc1nn2c(C)c(CN)c(C)nc2c1S(=O)(=O)c1cccc(Cl)c1,active,9.643974,1,0,0,0,1,0,0,0,0,0,1,0,0


In [24]:
X_train = df_train.drop(columns=meta_cols)
y_train = df_train["pIC50"]
X_test = df_test.drop(columns=meta_cols)
y_test = df_test["pIC50"]


In [25]:
X_train.head()

,fp_935,fp_883,fp_881,fp_879,fp_875,fp_867,fp_847,fp_843,fp_841,fp_885,fp_934,fp_932,fp_926
0,0,0,1,0,1,0,0,0,1,0,0,0,0
1,1,0,0,0,1,0,0,0,1,0,0,0,1
2,1,1,1,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,1,0,0,0,0,0,0,1,1
4,1,0,0,0,1,0,0,0,0,0,1,0,0


In [26]:

# === Models and Hyperparameters ===
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {"model__n_estimators": [100, 200], "model__max_depth": [None, 10, 20]}
    ),
    "XGBoost": (
        XGBRegressor(random_state=42, verbosity=0),
        {"model__n_estimators": [100, 200], "model__max_depth": [3, 6]}
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1]}
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {"model__n_neighbors": [3, 5, 7]}
    ),
    "SVR": (
        SVR(),
        {"model__C": [1, 10], "model__kernel": ["rbf", "linear"]}
    ),
    "MLPRegressor": (
        MLPRegressor(random_state=42, max_iter=500),
        {"model__hidden_layer_sizes": [(100,), (50, 50)], "model__alpha": [0.0001, 0.001]}
    ),
}

In [27]:
# === Optimization ===
results = []

for name, (model, param_grid) in models.items():
    print(f" Optimizing: {name}")

    # === Apply scaling if required ===
    if name in ["SVR", "KNeighborsRegressor", "MLPRegressor"]:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    else:
        pipeline = Pipeline([
            ("model", model)
        ])

    # === Grid search with 5-fold CV on training set ===
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2", n_jobs=-1)
    grid.fit(X_train, y_train)

    # === Evaluate on training set ===
    best_model = grid.best_estimator_
    y_pred_train = best_model.predict(X_train)
    train_r2 = r2_score(y_train, y_pred_train)
    train_rmse = sqrt(mean_squared_error(y_train, y_pred_train))

    # === Evaluate on test set ===
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))

    # === Store results ===
    results.append({
        "Target": target,
        "Model": name,
        "Best Params": grid.best_params_,
        "Train R2": round(train_r2, 4),
        "Train RMSE": round(train_rmse, 4),
        "Test R2": round(test_r2, 4),
        "Test RMSE": round(test_rmse, 4)
    })


 Optimizing: RandomForest
 Optimizing: XGBoost
 Optimizing: GradientBoosting
 Optimizing: KNeighborsRegressor
 Optimizing: SVR
 Optimizing: MLPRegressor


In [28]:
# === Show results ===
df_results = pd.DataFrame(results)
print("\ Hyperparameter Optimization Results:")
print(df_results)

\ Hyperparameter Optimization Results:
  Target                Model  \
0  5-HT6         RandomForest   
1  5-HT6              XGBoost   
2  5-HT6     GradientBoosting   
3  5-HT6  KNeighborsRegressor   
4  5-HT6                  SVR   
5  5-HT6         MLPRegressor   

                                         Best Params  Train R2  Train RMSE  \
0  {'model__max_depth': 10, 'model__n_estimators'...    0.2555      0.9232   
1  {'model__max_depth': 3, 'model__n_estimators':...    0.2309      0.9383   
2  {'model__learning_rate': 0.1, 'model__n_estima...    0.2039      0.9546   
3                          {'model__n_neighbors': 7}    0.1615      0.9797   
4            {'model__C': 1, 'model__kernel': 'rbf'}    0.2101      0.9509   
5  {'model__alpha': 0.0001, 'model__hidden_layer_...    0.2461      0.9290   

   Test R2  Test RMSE  
0   0.2258     0.8796  
1   0.2020     0.8930  
2   0.1908     0.8993  
3   0.0911     0.9531  
4   0.2341     0.8749  
5   0.1915     0.8989  


In [29]:
# === Save results ===

results_out = f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/Hyperparameter-optimization-traditional-ML/optimized_hyperparameters_traditional_{target}_topN.csv"
df_results.to_csv(results_out, index=False)

**Current Limitations / Issues**

- Overfitting Risk: The best hyperparameters are selected using the same training data used to assess performance via cross-validation. This can result in optimistically biased results.

- No Nested Cross-Validation: The current setup lacks an outer loop to independently validate the entire tuning process.

- Single Test Split: The test set is used once for final evaluation. Performance may depend on that split.

- Computational Load: Grid search over all models is computationally expensive, especially for large datasets or many features.